# 📚 Week 1: MNIST手写数字分类器

**目标**: 完整实现从数据加载到训练评估的流程

**里程碑**:
- 理解DataLoader机制
- 完成完整训练循环
- Loss曲线正常下降
- 测试准确率 > 95%

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 检查GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 数据加载与可视化

In [ ]:
# 数据预处理：转Tensor + 归一化
transform = transforms.Compose([
    transforms.ToTensor(),  # [0,255] -> [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST均值标准差
])

# 下载并加载MNIST数据集
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# 创建DataLoader
train_loader = DataLoader(
    train_dataset, 
    batch_size=64, 
    shuffle=True,  # 随机打乱
    num_workers=2   # 多进程加载
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=1000, 
    shuffle=False
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# 可视化MNIST样本
def show_samples(loader, num_images=10):
    examples = next(iter(loader))
    images, labels = examples[0][:num_images], examples[1][:num_images]
    
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flatten()):
        # 反归一化显示
        img = images[i].squeeze() 
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Label: {labels[i].item()}')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('mnist_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: mnist_samples.png")

show_samples(train_loader)

## 2. 定义模型

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()  # 28x28 -> 784
        self.fc = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Dropout(0.2),       # 防止过拟合
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)      # 10个数字类别
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)
        return x

model = MNISTNet().to(device)
print(model)

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

## 3. 训练与评估函数

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """训练一个epoch"""
    model.train()  # 设为训练模式
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)

        # 前向传播
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)

        # 反向传播 + 更新
        loss.backward()
        optimizer.step()

        # 统计
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    """评估模型"""
    model.eval()  # 设为评估模式
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # 不计算梯度
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

## 4. 开始训练

In [ ]:
# 超参数
EPOCHS = 10
LEARNING_RATE = 0.001

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 分类任务常用
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 记录训练历史
history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': []
}

print("Starting training...\n")
for epoch in range(EPOCHS):
    # 训练
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # 评估
    test_loss, test_acc = evaluate(
        model, test_loader, criterion, device
    )
    
    # 记录
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")

print("\nTraining completed!")

## 5. 可视化训练曲线

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss曲线
ax1.plot(history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(history['test_loss'], 'r-', label='Test Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Test Loss', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy曲线
ax2.plot(history['train_acc'], 'b-', label='Train Acc', linewidth=2)
ax2.plot(history['test_acc'], 'r-', label='Test Acc', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training & Test Accuracy', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Test Accuracy: {history['test_acc'][-1]:.2f}%")

## 6. 预测演示

In [ ]:
# 查看一些预测结果
model.eval()
examples = next(iter(test_loader))
images, labels = examples[0][:10], examples[1][:10]

with torch.no_grad():
    images_gpu = images.to(device)
    outputs = model(images_gpu)
    predictions = outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    img = images[i].squeeze()
    pred, label = predictions[i].item(), labels[i].item()
    color = 'green' if pred == label else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Pred: {pred} (True: {label})', color=color)
    ax.axis('off')
plt.tight_layout()
plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGreen = Correct, Red = Wrong")

## ✅ Week1 里程碑检查

- [ ] Colab GPU可用
- [ ] 完成Tensor操作练习
- [ ] 理解autograd工作原理
- [ ] 能独立写出nn.Module子类
- [ ] MNIST准确率 > 95%
- [ ] Loss曲线正常下降
- [ ] 生成3张图片保存

## 📤 保存模型
```python
# 保存整个模型
torch.save(model.state_dict(), 'mnist_model.pth')

# 加载模型
model = MNISTNet()
model.load_state_dict(torch.load('mnist_model.pth'))
```